# qlib_paper_sim_v1

SOXL qlib-backed production paper/sim 验收卡：先跑 qlib recorder/record，再从 qlib 预测 sidecar 生成 paper target-position intent。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow.record_temp import SignalRecord, SigAnaRecord, PortAnaRecord

root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root.parent == root:
        raise RuntimeError('repo root not found')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.workflow import QlibNativeExperimentSpec, run_qlib_native_experiment

provider_uri = root / 'data' / 'qlib' / 'soxl'
qlib.init(provider_uri=provider_uri.as_posix(), region='us')
_features = D.features(['SOXL'], ['$close'], start_time='2026-01-09', end_time='2026-01-15', freq='day')

card_dir = root / 'production' / 'qlib_paper_sim_v1'
spec = QlibNativeExperimentSpec(
    experiment='qlib_paper_sim_v1',
    provider_uri=provider_uri,
    recorder_uri=card_dir / 'qlib_recorder',
    output_dir=card_dir,
    market='all',
    region='us',
    instrument='SOXL',
    train=('2026-01-02', '2026-01-08'),
    test=('2026-01-09', '2026-01-15'),
)
result = run_qlib_native_experiment(spec)

predictions = pd.read_csv(card_dir / 'predictions.csv')
latest_score = float(predictions['score'].iloc[-1])
latest_price = float(_features['$close'].iloc[-1])
target_percent = 0.25 if latest_score > 0 else 0.0
quantity = int((10_000 * target_percent) / latest_price)
order_intents = pd.DataFrame([
    {
        'symbol': 'SOXL',
        'as_of': spec.test[1],
        'target_percent': target_percent,
        'quantity': quantity,
        'price': latest_price,
        'execution_mode': 'paper_sim',
        'real_broker_order': False,
        'recorder_id': result.recorder_id,
    }
])
positions = order_intents[['symbol', 'quantity']]
order_intents.to_csv(card_dir / 'order_intents.csv', index=False)
positions.to_csv(card_dir / 'positions.csv', index=False)
production_summary = {
    'strategy': 'qlib_paper_sim_v1',
    'execution_mode': 'paper_sim',
    'real_broker_orders': False,
    'provider_uri': result.summary['provider_uri'],
    'recorder_id': result.recorder_id,
    'order_intents': 'order_intents.csv',
    'positions': 'positions.csv',
    'qlib_native_api': result.summary['qlib_native_api'],
}
(card_dir / 'production_summary.json').write_text(json.dumps(production_summary, indent=2, sort_keys=True), encoding='utf-8')
production_summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[56246:MainThread](2026-06-22 14:47:08,330) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56246:MainThread](2026-06-22 14:47:08,332) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56246:MainThread](2026-06-22 14:47:08,332) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/qlib-native-spec-v1/data/qlib/soxl')}


[56246:MainThread](2026-06-22 14:47:08,338) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[56246:MainThread](2026-06-22 14:47:11,115) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56246:MainThread](2026-06-22 14:47:11,116) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56246:MainThread](2026-06-22 14:47:11,116) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/qlib-native-spec-v1/data/qlib/soxl')}


'The following are analysis results of benchmark return(1day).'
                        risk
mean                0.032403
std                 0.037493
annualized_return   7.711808
information_ratio  13.332671
max_drawdown       -0.012306
'The following are analysis results of the excess return without cost(1day).'
                       risk
mean              -0.018200
std                0.039281
annualized_return -4.331532
information_ratio -7.147842
max_drawdown      -0.014269
'The following are analysis results of the excess return with cost(1day).'
                       risk
mean              -0.018355
std                0.039196
annualized_return -4.368387
information_ratio -7.224143
max_drawdown      -0.014654
'The following are analysis results of indicators(1day).'
     value
ffr    1.0
pa     0.0
pos    0.0
[local] 读取 25 条数据


{'strategy': 'qlib_paper_sim_v1',
 'execution_mode': 'paper_sim',
 'real_broker_orders': False,
 'provider_uri': '/Users/rziiiii/dev/workspace/.worktrees/qlib-native-spec-v1/data/qlib/soxl',
 'recorder_id': 'f658122eb76a43db926ce666da586eec',
 'order_intents': 'order_intents.csv',
 'positions': 'positions.csv',
 'qlib_native_api': ['qlib.init',
  'qlib.data.D.features',
  'DatasetH',
  'DataHandlerLP',
  'Alpha158.get_feature_config',
  'qlib.workflow.R.start',
  'SignalRecord',
  'SigAnaRecord',
  'PortAnaRecord']}

## 中文结论

- 做了什么：使用 qlib-native reference 生成 recorder 和预测，再生成 paper/sim target-position intent。
- 核心指标：输出 order_intents.csv、positions.csv、production_summary.json、model_selection.csv、selected_model.json、risk_analysis.csv 和 backtest_benchmark_comparison.csv，并记录 qlib recorder id。
- regime 差异：本卡只负责 production paper/sim 合同，regime/window 对比由 analysis card 消费 recorder 结果、risk_analysis.csv 和 backtest_window_metrics.csv。
- 是否晋升：仅 paper/sim，不做真实 broker 下单。
- 风险：短窗口预测仅用于流程验收，不能作为交易信号。
- 下一步：在 schedule 中读取 recorder-registered manifest/export。